# 03 — Hugging Face parity

Load the HF `PekingU/rtdetr_r{18,50}vd` weights into **our** modules with `strict=True` and check that our outputs match HF's numerically (max abs diff < 1e-4).

Backbone done (Task 10). Encoder (Task 18) and end-to-end (Task 20) sections get added later.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import torch
import transformers
from transformers import RTDetrForObjectDetection
from src.backbone import PResNet

torch.manual_seed(0)
print("torch", torch.__version__, "| transformers", transformers.__version__)

/mnt/space/conda/envs/dlverse/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.13.0+cu126 | transformers 5.15.0.dev0


## 1. Backbone keys: ours vs. HF

HF backbone keys live under `model.backbone.model.`. After stripping that prefix they must equal our `PResNet` keys exactly (no remapping).

In [2]:
PREFIX = "model.backbone.model."

def hf_backbone_sd(hf):
    return {k[len(PREFIX):]: v for k, v in hf.state_dict().items() if k.startswith(PREFIX)}

hf = {d: RTDetrForObjectDetection.from_pretrained(f"PekingU/rtdetr_r{d}vd").eval() for d in (18, 50)}

for d in (18, 50):
    ref = hf_backbone_sd(hf[d])
    ours = PResNet(d, freeze_norm=True).state_dict()
    missing, extra = set(ref) - set(ours), set(ours) - set(ref)
    bad_shapes = [k for k in ref if k in ours and ref[k].shape != ours[k].shape]
    print(f"R{d}: {len(ours)} keys | missing {len(missing)} | extra {len(extra)} | shape mismatches {len(bad_shapes)}")
    assert not missing and not extra and not bad_shapes
print("✅ key sets and shapes identical")

Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 526/526 [00:00<00:00, 14423.69it/s]

Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 764/764 [00:00<00:00, 13357.32it/s]

R18: 115 keys | missing 0 | extra 0 | shape mismatches 0
R50: 275 keys | missing 0 | extra 0 | shape mismatches 0
✅ key sets and shapes identical


## 2. Strict load + C3–C5 parity

Same random `[1, 3, 640, 640]` input through ours and `hf.model.backbone.model` (returns `.feature_maps` = C3, C4, C5).

In [3]:
x = torch.randn(1, 3, 640, 640)
backbones = {}

for d in (18, 50):
    ours = PResNet(d, freeze_norm=True).eval()
    print(f"R{d}", ours.load_state_dict(hf_backbone_sd(hf[d]), strict=True))
    with torch.no_grad():
        ref = hf[d].model.backbone.model(x).feature_maps
        out = ours(x)
    for name, a, b in zip(("C3", "C4", "C5"), out, ref, strict=True):
        diff = (a - b).abs().max().item()
        print(f"   {name} {tuple(a.shape)}  max|diff| = {diff:.2e}")
        assert a.shape == b.shape and diff < 1e-4
    backbones[d] = ours
print("✅ backbone matches HF (R18 + R50)")

R18 <All keys matched successfully>


   C3 (1, 128, 80, 80)  max|diff| = 0.00e+00
   C4 (1, 256, 40, 40)  max|diff| = 0.00e+00
   C5 (1, 512, 20, 20)  max|diff| = 0.00e+00
R50 <All keys matched successfully>


   C3 (1, 512, 80, 80)  max|diff| = 0.00e+00
   C4 (1, 1024, 40, 40)  max|diff| = 0.00e+00
   C5 (1, 2048, 20, 20)  max|diff| = 0.00e+00
✅ backbone matches HF (R18 + R50)
